In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy

TIME_OFFSET = 10800 #ADT to UTC is 3hrs

In [6]:
def entropy_feature(x):
    counts = x.value_counts()
    if len(counts) <= 1:
        return 0.0
    return entropy(counts)

def coeff_var(x):
    mean = x.mean()
    if mean == 0:
        return 0.0
    return x.std() / mean

In [ ]:
import pandas as pd
import numpy as np

def process_file(k, input_csv, output_csv):
    print(f"Processing {input_csv}...")

    df = pd.read_csv(input_csv)
    numeric_cols = ["tcp.len", "mbtcp.len", "modbus.byte_cnt", "modbus.regval_uint16", 
                "modbus.reference_num", "modbus.response_time", "modbus.word_cnt",
                "modbus.byte_cnt", "tcp.analysis.retransmission"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["timestamp"] = df["frame.time_epoch"] + TIME_OFFSET
    df = df.sort_values("timestamp")
    df["time_window"] = (df["timestamp"] * k).astype(int)

    has_transaction_id = "mbtcp.trans_id" in df.columns

    # ── Precompute derived boolean/numeric columns ─────────────────────────────
    df["is_stacked"]   = df["tcp.len"] > (df["mbtcp.len"] + 6)
    df["stack_excess"] = (df["tcp.len"] - (df["mbtcp.len"] + 6)).clip(lower=0)
    df["len_mismatch"] = df["mbtcp.len"] - df["modbus.byte_cnt"]
    df["is_mismatch"]  = df["len_mismatch"].abs() > 2
    df["is_duplicate"] = df.duplicated(
        subset=["ip.src", "ip.dst", "mbtcp.trans_id", "modbus.data"], keep=False
    )
    df["tcp.analysis.retransmission"] = df["tcp.analysis.retransmission"].fillna(0).astype(int)

    # ── Fix IAT within (time_window, src, dst) ────────────────────────────────
    df["iat"] = (
        df.groupby(["time_window", "ip.src", "ip.dst"])["timestamp"]
        .diff()
        .fillna(0)
    )

    temp_src  = df["ip.src"].dropna().astype(str)
    temp_dst  = df["ip.dst"].dropna().astype(str)
    known_ips = sorted(set(temp_src.unique()) | set(temp_dst.unique()))
    print(f"IPs found: {known_ips}")

    observed_windows = pd.Index(sorted(df["time_window"].unique()), name="time_window")

    # ── Per-IP feature extraction ──────────────────────────────────────────────
    ip_frames = []

    for ip in known_ips:
        tx = df[df["ip.src"] == ip]
        rx = df[df["ip.dst"] == ip]

        tx_agg = tx.groupby("time_window").agg(
            # Volume
            tx_packet_count       = ("frame.len",                "count"),
            tx_total_bytes        = ("frame.len",                "sum"),
            tx_mean_pkt_size      = ("frame.len",                "mean"),
            tx_std_pkt_size       = ("frame.len",                "std"),
            # IAT
            tx_iat_mean           = ("iat",                      "mean"),
            tx_iat_std            = ("iat",                      "std"),
            tx_iat_cv             = ("iat",                      coeff_var),
            # Destinations & slaves
            tx_unique_dst         = ("ip.dst",                   "nunique"),
            tx_unique_unit_ids    = ("mbtcp.unit_id",            "nunique"),
            # Function codes
            tx_unique_fc          = ("modbus.func_code",         "nunique"),
            tx_func_entropy       = ("modbus.func_code",         entropy_feature),
            tx_read_count         = ("modbus.func_code",         lambda x: x.isin([3, 4]).sum()),
            tx_write_count        = ("modbus.func_code",         lambda x: x.isin([5, 6, 15, 16]).sum()),
            tx_fc3                = ("modbus.func_code",         lambda x: (x == 3).sum()),
            tx_fc4                = ("modbus.func_code",         lambda x: (x == 4).sum()),
            tx_fc5                = ("modbus.func_code",         lambda x: (x == 5).sum()),
            tx_fc6                = ("modbus.func_code",         lambda x: (x == 6).sum()),
            tx_fc15               = ("modbus.func_code",         lambda x: (x == 15).sum()),
            tx_fc16               = ("modbus.func_code",         lambda x: (x == 16).sum()),
            # Exceptions
            tx_exception_count    = ("modbus.exception_code",    lambda x: x.notna().sum()),
            # Register addressing
            tx_unique_regs        = ("modbus.reference_num",     "nunique"),
            tx_register_std       = ("modbus.reference_num",     "std"),
            tx_register_entropy   = ("modbus.reference_num",     entropy_feature),
            # Register values
            tx_regval_mean        = ("modbus.regval_uint16",     "mean"),
            tx_regval_std         = ("modbus.regval_uint16",     "std"),
            # Payload sizes
            tx_mean_mbap_len      = ("mbtcp.len",                "mean"),
            tx_byte_cnt_mean      = ("modbus.byte_cnt",          "mean"),
            tx_word_cnt_mean      = ("modbus.word_cnt",          "mean"),
            # Response timing
            tx_resp_time_mean     = ("modbus.response_time",     "mean"),
            tx_resp_time_std      = ("modbus.response_time",     "std"),
            # Retransmissions
            tx_retransmission_sum = ("tcp.analysis.retransmission", "sum"),
            # Stacking
            tx_stacked_count      = ("is_stacked",               "sum"),
            tx_mean_stack_excess  = ("stack_excess",             "mean"),
            # Length mismatch
            tx_len_mismatch_mean  = ("len_mismatch",             "mean"),
            tx_len_mismatch_count = ("is_mismatch",              "sum"),
            # Duplicate / replay
            tx_duplicate_count    = ("is_duplicate",             "sum"),
        )

        rx_agg = rx.groupby("time_window").agg(
            rx_packet_count       = ("frame.len",                "count"),
            rx_total_bytes        = ("frame.len",                "sum"),
            rx_unique_src         = ("ip.src",                   "nunique"),
            rx_unique_unit_ids    = ("mbtcp.unit_id",            "nunique"),
            rx_retransmission_sum = ("tcp.analysis.retransmission", "sum"),
            rx_resp_time_mean     = ("modbus.response_time",     "mean"),
        )

        if has_transaction_id:
            tx_tid = tx.groupby("time_window").agg(
                tx_unique_tids = ("mbtcp.trans_id", "nunique")
            )
            tx_agg = tx_agg.join(tx_tid, how="left")

        tx_agg = tx_agg.reindex(observed_windows, fill_value=0)
        rx_agg = rx_agg.reindex(observed_windows, fill_value=0)

        ip_df = tx_agg.join(rx_agg, how="left").fillna(0)

        # ── Derived ratios ─────────────────────────────────────────────────────
        pkt = ip_df["tx_packet_count"].to_numpy(dtype=float)

        def safe_div(num_col):
            num = ip_df[num_col].to_numpy(dtype=float)
            return np.divide(num, pkt, out=np.zeros_like(pkt), where=pkt > 0)

        ip_df["tx_write_ratio"]     = safe_div("tx_write_count")
        ip_df["tx_read_ratio"]      = safe_div("tx_read_count")
        ip_df["tx_exception_ratio"] = safe_div("tx_exception_count")

# and for the tid ratio:
        if has_transaction_id:
            ip_df["tx_dup_tids"]  = ip_df["tx_packet_count"] - ip_df["tx_unique_tids"]
            ip_df["tx_tid_ratio"] = safe_div("tx_unique_tids")
        else:
            ip_df["tx_unique_tids"] = 0
            ip_df["tx_dup_tids"]    = 0
            ip_df["tx_tid_ratio"]   = 0.0

        ip_df["tx_regval_delta"] = ip_df["tx_regval_mean"].diff().fillna(0)

        # ── Prefix with IP ─────────────────────────────────────────────────────
        safe_ip = ip.replace(".", "_")
        ip_df   = ip_df.add_prefix(f"{safe_ip}__")

        ip_frames.append(ip_df)

    # ── Join all IPs into one row per time_window ──────────────────────────────
    result = pd.DataFrame(index=observed_windows)
    for ip_df in ip_frames:
        result = result.join(ip_df, how="left")

    result = result.fillna(0).reset_index()

    packet_counts = df.groupby("time_window").size()
    print("Window packet count description:")
    print(packet_counts.describe())

    result.to_csv(output_csv, index=False)
    print(f"Saved → {output_csv}\n")

In [8]:
process_file(1, "../train/benign_nw_analysis.csv", "../train/1s_benign_flows.csv")

Processing ../train/benign_nw_analysis.csv...


/tmp/ipykernel_471138/2290100831.py:7: DtypeWarning: Columns (0: tcp.analysis.duplicate_ack, 1: modbus.data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


IPs found: ['185.175.0.1', '185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.8', '224.0.0.22', '224.0.0.251']


/tmp/ipykernel_471138/2290100831.py:143: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count    7085.000000
mean       75.892025
std        90.031639
min         1.000000
25%        50.000000
50%        50.000000
75%        90.000000
max      1292.000000
dtype: float64
Saved → ../train/1s_benign_flows.csv



In [15]:
process_file(1, "../train/cscada_attack_ssw_analysis.csv", "../train/1s_cscada_flows.csv")

Processing ../train/cscada_attack_ssw_analysis.csv...


/tmp/ipykernel_471138/3647072646.py:7: DtypeWarning: Columns (0: mbtcp.trans_id, 1: mbtcp.unit_id, 2: mbtcp.len, 3: mbtcp.prot_id, 4: modbus.func_code, 5: modbus.reference_num, 6: modbus.regnum16, 7: modbus.bitnum, 8: modbus.word_cnt, 9: modbus.bit_cnt, 10: modbus.regval_uint16, 11: modbus.data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


IPs found: ['185.175.0.1', '185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.8', '224.0.0.251']


/tmp/ipykernel_471138/3647072646.py:157: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count    14981.000000
mean        77.228289
std        144.291295
min          1.000000
25%         45.000000
50%         50.000000
75%         90.000000
max       8241.000000
dtype: float64
Saved → ../train/1s_cscada_flows.csv



In [17]:
process_file(1, "../train/ext_attack_nw_analysis.csv", "../train/1s_external_flows.csv")

Processing ../train/ext_attack_nw_analysis.csv...


/tmp/ipykernel_471138/3647072646.py:7: DtypeWarning: Columns (0: tcp.analysis.duplicate_ack, 1: modbus.data) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv)


IPs found: ['185.175.0.1', '185.175.0.3', '185.175.0.4', '185.175.0.5', '185.175.0.6', '185.175.0.7', '185.175.0.8', '224.0.0.22', '224.0.0.251']


/tmp/ipykernel_471138/3647072646.py:157: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  result = result.fillna(0).reset_index()


Window packet count description:
count     1881.000000
mean       290.755981
std       1252.626481
min          1.000000
25%         45.000000
50%         45.000000
75%         51.000000
max      20281.000000
dtype: float64
Saved → ../train/1s_external_flows.csv



In [ ]:
df = pd.read_csv("../train/1s_cscada_flows.csv")
print(df.columns)

Index(['time_window', '185_175_0_1__tx_packet_count',
       '185_175_0_1__tx_total_bytes', '185_175_0_1__tx_mean_pkt_size',
       '185_175_0_1__tx_std_pkt_size', '185_175_0_1__tx_unique_dst',
       '185_175_0_1__tx_iat_mean', '185_175_0_1__tx_iat_std',
       '185_175_0_1__tx_iat_cv', '185_175_0_1__tx_burstiness',
       ...
       '224_0_0_251__rx_packet_count', '224_0_0_251__rx_total_bytes',
       '224_0_0_251__rx_unique_src', '224_0_0_251__tx_write_ratio',
       '224_0_0_251__tx_read_ratio', '224_0_0_251__tx_exception_ratio',
       '224_0_0_251__tx_unique_tids', '224_0_0_251__tx_dup_tids',
       '224_0_0_251__tx_tid_ratio', '224_0_0_251__tx_regval_delta'],
      dtype='str', length=267)


In [ ]:
# process_file(10, "../train/benign_nw.csv", "../train/100ms_benign_flows.csv") #this is for one second windows

In [ ]:
# process_file(10, "../train/cscada_attack_ssw.csv", "../train/100ms_cscada_flows.csv")

In [ ]:
# process_file(10, "../train/ext_attack_nw.csv", "../train/100ms_external_flows.csv")

In [ ]:
df = pd.read_csv("../train/1s_cscada_flows.csv")
for col in df.columns:
    print("\"" + col + "\",", end=" ")

"time_window", "185_175_0_1__tx_packet_count", "185_175_0_1__tx_total_bytes", "185_175_0_1__tx_mean_pkt_size", "185_175_0_1__tx_std_pkt_size", "185_175_0_1__tx_unique_dst", "185_175_0_1__tx_iat_mean", "185_175_0_1__tx_iat_std", "185_175_0_1__tx_iat_cv", "185_175_0_1__tx_burstiness", "185_175_0_1__tx_unique_fc", "185_175_0_1__tx_func_entropy", "185_175_0_1__tx_read_count", "185_175_0_1__tx_write_count", "185_175_0_1__tx_fc3", "185_175_0_1__tx_fc4", "185_175_0_1__tx_fc6", "185_175_0_1__tx_fc16", "185_175_0_1__tx_exception_count", "185_175_0_1__tx_unique_regs", "185_175_0_1__tx_register_mean", "185_175_0_1__tx_register_std", "185_175_0_1__tx_register_entropy", "185_175_0_1__tx_regval_mean", "185_175_0_1__tx_regval_std", "185_175_0_1__tx_word_cnt_mean", "185_175_0_1__tx_byte_cnt_mean", "185_175_0_1__tx_resp_time_mean", "185_175_0_1__tx_resp_time_std", "185_175_0_1__rx_packet_count", "185_175_0_1__rx_total_bytes", "185_175_0_1__rx_unique_src", "185_175_0_1__tx_write_ratio", "185_175_0_1__tx